# Polymarket Crypto Dataset Check

Lightweight validation notebook for the BTC/ETH core plus crypto policy/ETF/MicroStrategy dataset. This notebook does not run PCA, persistent homology, forecasting models, or neural nets.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = ROOT / "data" / "processed"

markets = pd.read_parquet(PROCESSED / "market_universe.parquet")
selected = pd.read_csv(PROCESSED / "selected_markets.csv")
prices = pd.read_parquet(PROCESSED / "prices_long.parquet")
core_panel = pd.read_parquet(PROCESSED / "panel_hourly_core.parquet")
core_plus_panel = pd.read_parquet(PROCESSED / "panel_hourly_core_plus_satellites.parquet")
coverage_market = pd.read_csv(PROCESSED / "coverage_by_market.csv")
coverage_time = pd.read_csv(PROCESSED / "coverage_by_timestamp.csv")
validation = json.loads((PROCESSED / "validation_report.json").read_text())
manifest = json.loads((PROCESSED / "dataset_manifest.json").read_text())
prices["timestamp"] = pd.to_datetime(prices["timestamp"], utc=True)
core_panel.index = pd.to_datetime(core_panel.index, utc=True)
core_plus_panel.index = pd.to_datetime(core_plus_panel.index, utc=True)

## Validation Status

In [ ]:
print("analysis_ready:", validation["analysis_ready"])
print("failed gates:", [k for k, v in validation["quality_gates"].items() if not v])
print(json.dumps(validation["coverage"], indent=2))
print("limitations:")
for item in validation["limitations"]:
    print("-", item)

## Market Counts

In [ ]:
summary = {
    "metadata_markets": len(markets),
    "selected_markets": len(selected),
    "core_markets": int(selected["is_core"].sum()),
    "satellite_markets": int(selected["is_satellite"].sum()),
    "price_rows": len(prices),
    "price_markets": prices["market_id"].nunique(),
    "core_panel_shape": core_panel.shape,
    "core_plus_panel_shape": core_plus_panel.shape,
}
summary

In [ ]:
selected["market_family"].value_counts().rename_axis("market_family").reset_index(name="markets")

## Coverage Quality

In [ ]:
coverage_market.sort_values("usable_points").head(10)[[
    "market_id", "question", "market_family", "asset", "usable_points", "observed_days", "timestamp_min", "timestamp_max"
]]

In [ ]:
coverage_market["usable_points"].describe()

In [ ]:
coverage_time[["active_markets", "missing_fraction", "active_core_markets"]].describe()

## Panel Missingness

In [ ]:
pd.DataFrame({
    "panel": ["core", "core_plus_satellites"],
    "rows": [core_panel.shape[0], core_plus_panel.shape[0]],
    "markets": [core_panel.shape[1], core_plus_panel.shape[1]],
    "missingness": [core_panel.isna().mean().mean(), core_plus_panel.isna().mean().mean()],
    "timestamp_min": [core_panel.index.min(), core_plus_panel.index.min()],
    "timestamp_max": [core_panel.index.max(), core_plus_panel.index.max()],
})

## Sample Price Histories

In [ ]:
sample_ids = (
    coverage_market[coverage_market["is_core"].fillna(False)]
    .sort_values("usable_points", ascending=False)
    .head(5)["market_id"]
    .astype(str)
    .tolist()
)
plot_df = prices[prices["market_id"].astype(str).isin(sample_ids)].copy()
labels = selected.set_index(selected["market_id"].astype(str))["question"].to_dict()
fig, ax = plt.subplots(figsize=(12, 6))
for market_id, group in plot_df.groupby(plot_df["market_id"].astype(str)):
    group = group.sort_values("timestamp")
    ax.plot(group["timestamp"], group["yes_price"], label=labels.get(market_id, market_id)[:60])
ax.set_title("Sample Core Market YES Price Histories")
ax.set_xlabel("Timestamp")
ax.set_ylabel("YES price")
ax.set_ylim(-0.02, 1.02)
ax.legend(loc="best", fontsize=8)
fig.autofmt_xdate()
plt.show()

## Stress Tests

In [ ]:
stress_path = PROCESSED / "stress_tests" / "stress_test_summary.csv"
if stress_path.exists():
    stress = pd.read_csv(stress_path)
    display(stress)
else:
    print("Run python src/stress_test_dataset.py to create stress-test outputs.")